Chargement des données

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# Route exacte fournie
DOSSIER_SPECTROGRAMMES = (
    r"D:\Personel\HEPL\M2Q1\Projet IA\Projet\AI-Project\Spectogramme"
)

print(f"Dossier cible utilisé : {DOSSIER_SPECTROGRAMMES}")

IMG_SIZE = 128
X = []
y = []
mapping_classes = {"engine1_good": 0, "engine2_broken": 1, "engine3_heavyload": 2}

print("Chargement des images en cours...")
if os.path.exists(DOSSIER_SPECTROGRAMMES):
  for fichier in os.listdir(DOSSIER_SPECTROGRAMMES):
    if fichier.endswith(".png"):
      classe_trouvee = None
      for nom_classe in mapping_classes.keys():
        if nom_classe in fichier:
          classe_trouvee = mapping_classes[nom_classe]
          break
      if classe_trouvee is not None:
        chemin = os.path.join(DOSSIER_SPECTROGRAMMES, fichier)
        img = cv2.imread(chemin)
        if img is not None:
          img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
          img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
          X.append(img)
          y.append(classe_trouvee)

  X = np.array(X, dtype="float32") / 255.0
  y = np.array(y)

  if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(
        f"Données chargées avec succès ! Train: {len(X_train)} images | Test:"
        f" {len(X_test)} images"
    )
  else:
    print(
        "Attention : Le dossier existe mais aucune image ne correspond aux"
        " classes recherchées."
    )
else:
  print("Erreur : Le chemin spécifié est introuvable sur le disque.")

Dossier cible utilisé : D:\Personel\HEPL\M2Q1\Projet IA\Projet\AI-Project
Chargement des images en cours...


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 1. Modèle de Référence (Baseline) : Régression Logistique
Comme l'exige le cahier des charges, nous commençons par implémenter un modèle simple pour établir une base de comparaison. Les images sont aplaties en vecteurs 1D.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

modele_simple = LogisticRegression(max_iter=1000)
modele_simple.fit(X_train_flat, y_train)

y_pred_simple = modele_simple.predict(X_test_flat)
acc_simple = accuracy_score(y_test, y_pred_simple)
print(f"Taux de détection (Modèle Simple) : {acc_simple*100:.2f}%\n")
print(
    classification_report(
        y_test, y_pred_simple, target_names=["Sain", "Cassé", "Forte Charge"]
    )
)

## 2. Modèle Avancé 1 : Réseau de Neurones Convolutif (CNN)
Le CNN est particulièrement adapté pour traiter les spectrogrammes (images) car il permet d'extraire automatiquement les motifs visuels et les fréquences harmoniques caractéristiques des défauts mécaniques du moteur.

In [ ]:
from tensorflow.keras import layers, models

cnn_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax"),
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history = cnn_model.fit(
    X_train, y_train, epochs=10, validation_data=(X_test, y_test), batch_size=32
)

loss, acc_cnn = cnn_model.evaluate(X_test, y_test, verbose=0)
print(f"\nTaux de détection final (CNN) : {acc_cnn*100:.2f}%")